Baseline models:

1-D CNN

LSTM 

maybe XGBOOST

In [1]:
print("hello world")

hello world


In [2]:
!pip install vitaldb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 91.3 MB/s eta 0:00:00:00:010:01
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pan

In [18]:
import pandas as pd
import numpy as np
import time
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


import vitaldb as v

# 1-D CNN

Take raw EEG and apply Z-score normalization

In [4]:
track_names = ["SNUADC/ART",        # Arterial pressure wave  | W/500 | mmHg
               "SNUADC/ECG_II",     # ECG lead II wave        | W/500 | mV
               "SNUADC/ECG_V5",     # ECG lead V5 wave        | W/500 | mV
               "BIS/EEG1_WAV",      # EEG wave from channel 1 | W/128 | uV
               "BIS/EEG2_WAV",      # EEG wave from channel 2 | W/128 | uV
               "Solar8000/RR_CO2",  # Respiratory rate based on capnography | N | /min
               "Primus/CO2",        # Capnography wave        | W/62.5 | mmHg
               "BIS/BIS",           # Bispectral index value  |    N   | unitless
               ]                

# read 1 patient from the VitalDB server
patient = v.VitalFile(1, track_names)  # should we add max length?
# convert to pandas dataframe
patient = patient.to_pandas(track_names=track_names, interval=5)
# change column names to 'human-readable'
patient.columns = ['arterial_pres', 'ecg1', 'ecg2', 'eeg1', 'eeg2', 'resp_rate', 'capnography', 'bis']

### PyTorch

In [ ]:
class TimeSeriesCNN(nn.Module):
    def __init__(self, num_features, seq_len):
        super(TimeSeriesCNN, self).__init__()

        # 1st Layer - small, local patterns
        self.conv1 = nn.Conv1d(in_channels=num_features, out_channels=32, kernel_size=5, padding=2)
        # Pooling - shrinks timeline
        self.pool1 = nn.MaxPool1d(kernel_size=2)

        self.relu = nn.ReLU()

        # 2nd layer - sees 'zoomed-out' patterns (the receptive field)
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool1d(kernel_size=2)

        # Fully connected layers
        self.flatten_size = 64 * (seq_len // 4)
        self.fc1 = nn.Linear(self.flatten_size, 128)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool1(x)
        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool2(x)

        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

In [19]:
epochs = 20
batch_size = 32
seq_len = 200
split = 0.8
device = 'cuda'


patient = patient[['eeg1', 'eeg2', 'bis']].dropna()
patient = patient[patient['bis'] > 0]
patient['eeg1'] = (patient['eeg1'] - patient['eeg1'].mean()) / patient['eeg1'].std()
patient['eeg2'] = (patient['eeg2'] - patient['eeg2'].mean()) / patient['eeg2'].std()
x = np.stack([patient.eeg1, patient.eeg2], axis=1)
X = np.array([x[i:i+seq_len] for i in range(x.shape[0]-seq_len)])
X = X.reshape(-1, seq_len, 2)
X = X.transpose(0,2,1)

y = patient.bis
y = (y - np.mean(y)) / np.std(y)
Y = np.array([y[i+seq_len] for i in range(len(y)-seq_len)]).reshape(-1, 1)

split_idx = int(len(X) * split)
X_train = X[:split_idx]
Y_train = Y[:split_idx]

model = TimeSeriesCNN(num_features=2, seq_len=seq_len)
model.to(device)
model.train()

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

dataset_train = TensorDataset(torch.tensor(X_train, dtype=torch.float32).to(device), 
                        torch.tensor(Y_train, dtype=torch.float32).to(device))
dataloader_train = DataLoader(dataset_train, batch_size=batch_size)

start = time.time()
for epoch in range(epochs):
    for batch_x, batch_y in dataloader_train:
        optimizer.zero_grad()
        y_hat = model(batch_x)
        loss = criterion(y_hat, batch_y)
        loss.backward()
        optimizer.step()
    
    print(f"Epoch {epoch+1}/{epochs}: Loss={loss.item():.6f}")
end = time.time()
print(f"Total training time: {end-start}")

Epoch 1/20: Loss=0.240937
Epoch 2/20: Loss=0.112608
Epoch 3/20: Loss=0.104462
Epoch 4/20: Loss=0.100747
Epoch 5/20: Loss=0.097198
Epoch 6/20: Loss=0.241066
Epoch 7/20: Loss=0.094648
Epoch 8/20: Loss=0.106598
Epoch 9/20: Loss=0.264196
Epoch 10/20: Loss=0.123108
Epoch 11/20: Loss=0.121421
Epoch 12/20: Loss=0.091380
Epoch 13/20: Loss=0.121233
Epoch 14/20: Loss=0.090069
Epoch 15/20: Loss=0.106524
Epoch 16/20: Loss=0.152177
Epoch 17/20: Loss=0.134869
Epoch 18/20: Loss=0.102007
Epoch 19/20: Loss=0.141290
Epoch 20/20: Loss=0.115197
Total training time: 3.4397270679473877


### Braindecode